# 02 - Build container image and configure the Function App

This notebook:
1. Builds the Function App container image in ACR via `az acr build` (no local Docker required).
2. Forces the Function App to pull the freshly built image and restarts it.
3. Retrieves the function host key and writes it to `env/.env` as `AZURE_FUNCTION_KEY`.

**Pre-requisite:** `01_deploy_infra.ipynb` has completed and `env/.env` exists.

**Estimated time:** 5-8 minutes (first image build is the slowest).

---

## Step 1 - Load environment

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

load_dotenv(dotenv_path="../env/.env")

required = [
    "AZURE_SUBSCRIPTION_ID",
    "AZURE_RESOURCE_GROUP",
    "AZURE_ACR_NAME",
    "AZURE_CONTAINER_IMAGE_NAME",
    "AZURE_CONTAINER_IMAGE_REFERENCE",
    "AZURE_FUNCTION_APP_NAME",
]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing environment variables (run notebook 01 first): {missing}")

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install it or add it to PATH.")

subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]
resource_group = os.environ["AZURE_RESOURCE_GROUP"]
acr_name = os.environ["AZURE_ACR_NAME"]
image_name = os.environ["AZURE_CONTAINER_IMAGE_NAME"]
image_reference = os.environ["AZURE_CONTAINER_IMAGE_REFERENCE"]
function_app_name = os.environ["AZURE_FUNCTION_APP_NAME"]
image_tag = image_reference.split(":")[-1]

print(f"Resource group : {resource_group}")
print(f"ACR            : {acr_name}")
print(f"Image          : {image_reference}")
print(f"Function app   : {function_app_name}")

## Step 2 - Build the container image in ACR

`az acr build` builds the image in Azure using the Dockerfile in `../functionapp/`. No local Docker daemon is required.

This step installs `git`, Node.js 20, and `@github/copilot` into the image and typically takes 3-5 minutes on the first build.

In [ ]:
functionapp_dir = Path("../functionapp").resolve()
if not (functionapp_dir / "Dockerfile").exists():
    raise RuntimeError(f"Dockerfile not found in {functionapp_dir}")

build_result = subprocess.run(
    [
        az_cmd,
        "acr",
        "build",
        "--registry",
        acr_name,
        "--image",
        f"{image_name}:{image_tag}",
        str(functionapp_dir),
    ],
    capture_output=True,
    text=True,
)
print(build_result.stdout)
if build_result.returncode != 0:
    raise RuntimeError(
        f"az acr build failed (exit {build_result.returncode})\n"
        f"stderr: {build_result.stderr.strip()}\n"
        f"stdout: {build_result.stdout.strip()}"
    )
print(f"Image built and pushed: {image_reference}")

## Step 3 - Point the Function App at the freshly built image and restart

The Bicep template already set `linuxFxVersion` to the target image reference, but on the first deployment the tag did not exist yet. Setting the container config again (with MI creds) forces the Function App to re-pull the image.

In [ ]:
set_image_result = subprocess.run(
    [
        az_cmd,
        "functionapp",
        "config",
        "container",
        "set",
        "--name",
        function_app_name,
        "--resource-group",
        resource_group,
        "--image",
        image_reference,
    ],
    capture_output=True,
    text=True,
)
if set_image_result.returncode != 0:
    raise RuntimeError(
        f"az functionapp config container set failed (exit {set_image_result.returncode})\n"
        f"stderr: {set_image_result.stderr.strip()}\n"
        f"stdout: {set_image_result.stdout.strip()}"
    )
print("Container image reference updated on the Function App.")

restart_result = subprocess.run(
    [
        az_cmd,
        "functionapp",
        "restart",
        "--name",
        function_app_name,
        "--resource-group",
        resource_group,
    ],
    capture_output=True,
    text=True,
)
if restart_result.returncode != 0:
    raise RuntimeError(
        f"az functionapp restart failed (exit {restart_result.returncode})\n"
        f"stderr: {restart_result.stderr.strip()}"
    )
print("Function App restart triggered.")

## Step 4 - Wait for the Function App to become Running

Polls the app state and the registered functions list. The `CopilotReview` function should appear once the container image is pulled and the Python worker imports `function_app.py`.

In [ ]:
import json
import time

import requests
from azure.identity import AzureCliCredential

# process_timeout defaults to 10s, which is too tight on Windows: az.cmd
# cold-start plus token acquisition regularly exceeds 10s and surfaces as
# CredentialUnavailableError ("Failed to invoke the Azure CLI").
credential = AzureCliCredential(process_timeout=60)
management_token = credential.get_token("https://management.azure.com/.default").token
management_headers = {"Authorization": f"Bearer {management_token}"}

base_site_url = (
    f"https://management.azure.com/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}/providers/Microsoft.Web/sites/{function_app_name}"
)


def arm_request(method: str, relative_path: str, payload: dict | None = None):
    response = requests.request(
        method=method,
        url=f"{base_site_url}{relative_path}",
        headers=management_headers,
        params={"api-version": "2022-03-01"},
        json=payload,
        timeout=60,
    )
    if response.status_code < 200 or response.status_code >= 300:
        raise RuntimeError(
            f"ARM {method} {relative_path} failed with HTTP {response.status_code}.\n"
            f"Response body: {response.text}"
        )
    if response.text.strip():
        return response.json()
    return {}


# Poll for Running state, up to ~5 minutes.
for attempt in range(30):
    site = arm_request("GET", "")
    state = site.get("properties", {}).get("state", "")
    print(f"[{attempt + 1:>2}/30] state={state}")
    if state == "Running":
        break
    time.sleep(10)
else:
    raise RuntimeError("Function App did not reach Running state within 5 minutes.")

# Poll for the function to register.
for attempt in range(18):
    functions_data = arm_request("GET", "/functions")
    function_names = [fn.get("name", "") for fn in functions_data.get("value", [])]
    if function_names:
        print(f"Registered functions: {function_names}")
        break
    print(f"[{attempt + 1:>2}/18] waiting for function registration...")
    time.sleep(10)
else:
    raise RuntimeError(
        "Container is Running but no functions registered. Check container logs:\n"
        f"  az webapp log tail --name {function_app_name} --resource-group {resource_group}"
    )


## Step 5 - Retrieve the function key and write it to `env/.env`

In [ ]:
from dotenv import set_key

# The Function App uses MI-only storage. On first run there is typically a
# 1-5 minute lag before the host can read/write the master key in the
# azure-webjobs-secrets blob container. During that window /listKeys returns
# HTTP 400 wrapping "Encountered an error (InternalServerError) from host runtime".
MAX_LISTKEYS_ATTEMPTS = 30
LISTKEYS_BACKOFF_SECONDS = 10

host_keys = None
for attempt in range(1, MAX_LISTKEYS_ATTEMPTS + 1):
    try:
        host_keys = arm_request("POST", "/host/default/listKeys")
        break
    except RuntimeError as err:
        msg = str(err)
        transient = (
            "HTTP 400" in msg and "InternalServerError" in msg and "host runtime" in msg
        )
        if not transient or attempt == MAX_LISTKEYS_ATTEMPTS:
            raise
        print(
            f"[{attempt:>2}/{MAX_LISTKEYS_ATTEMPTS}] listKeys transient host error "
            f"(likely MI role propagation) — retrying in {LISTKEYS_BACKOFF_SECONDS}s..."
        )
        time.sleep(LISTKEYS_BACKOFF_SECONDS)

function_app_key = host_keys.get("functionKeys", {}).get("default") or host_keys.get("masterKey")
if not function_app_key:
    raise RuntimeError("ARM returned an empty function app host key.")

env_path = Path("../env/.env")
if not env_path.exists():
    raise RuntimeError("Expected ../env/.env to exist. Rerun notebook 01 first.")

set_key(str(env_path), "AZURE_FUNCTION_KEY", function_app_key, quote_mode="never")
print(f"Function key written to {env_path.resolve()}")
print("\nReady for notebook 03.")


---

## Configuration complete

Proceed to **`03_test.ipynb`** to trigger a Copilot CLI review from a notebook button.